In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import tune
from sklearn.utils import resample
from sklearn.preprocessing import OrdinalEncoder

import os
from sklearn.model_selection import cross_val_score
from dotenv import load_dotenv, dotenv_values 
# loading variables from .env file
load_dotenv() 

import mlflow

mlflow.set_tracking_uri(os.getenv('MLFLOW_SERVER'))
mlflow.set_experiment("Kaggle-s6e5")

In [ ]:
pipe, x, y = tune.process_data()

In [ ]:
model_name_hgb = "Best_HGB_s6ep5"
model_name_lgbm = "Best_LGBM_s6ep5"
model_name_cb = "Best_CB_s6ep5"
model_version = "latest"

# # Load the model from the Model Registry
model_uri_hgb = f"models:/{model_name_hgb}/{model_version}"
model_uri_lgbm = f"models:/{model_name_lgbm}/{model_version}"
model_uri_cb = f"models:/{model_name_cb}/{model_version}"

model_hgb = mlflow.sklearn.load_model(model_uri_hgb)
model_lgbm = mlflow.sklearn.load_model(model_uri_lgbm)
model_cb = mlflow.sklearn.load_model(model_uri_cb)

# model.fit(x, y)

In [ ]:
from sklearn.pipeline import Pipeline

p1 = Pipeline(steps=[('preprocessor', pipe), ('model', model_hgb)])
p2 = Pipeline(steps=[('preprocessor', pipe), ('model', model_lgbm)])
p3 = Pipeline(steps=[('preprocessor', pipe), ('model', model_cb)])

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(estimators=[('hgb', p1), ('lgbm', p2), ('cb', p3)], voting='soft')
#cv_scores = cross_val_score(voting_clf, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")
voting_clf.fit(x, y)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier

base_models = [('hgb', p1), ('lgbm', p2), ('cb', p3)]
meta_model = LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced", random_state=1)
#meta_model = DecisionTreeClassifier(max_depth=3, random_state=1)
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
stacking_clf.fit(x, y)

In [ ]:
with mlflow.start_run(nested=False, run_name=f"trial_{model_name_hgb}_{model_name_lgbm}_{model_name_cb}_stacking"):
    # mlflow.log_param("model_hgb", model_name_hgb)
    # mlflow.log_param("model_lgbm", model_name_lgbm)
    # mlflow.log_param("model_cb", model_name_cb)
    # mlflow.log_param("meta_model", "LogisticRegression")
    mlflow.log_param("params", stacking_clf.get_params(deep=True))
    #mlflow.sklearn.log_model(voting_clf, "voting_clf")
    mlflow.sklearn.log_model(stacking_clf, "stacking_clf")
    mlflow.log_metric("cv_score", np.mean(cross_val_score(stacking_clf, x, y, cv=5, scoring='roc_auc', error_score='raise')))

In [ ]:
testing_data = pd.read_csv("./data/test.csv")
#preds_voting = voting_clf.predict_proba(testing_data)[:, 1]
preds_stacking = stacking_clf.predict_proba(testing_data)[:, 1]



In [ ]:
submission = pd.DataFrame({
    "id": testing_data["id"],
    "PitNextLap": preds_voting
})

out_dir = 'data/'

submission.to_csv(os.path.join(out_dir, 'voting_clf_tuned_v1.csv'), index=False)

In [ ]:
submission_stacking = pd.DataFrame({
    "id": testing_data["id"],
    "PitNextLap": preds_stacking
})

out_dir = 'data/'

submission_stacking.to_csv(os.path.join(out_dir, 'stacking_clf_tuned_v2.csv'), index=False)